# Alzheimer's OASIS-1 — 标签整合 + 特征提取质量验证

承接 `alzheimer_feature_extraction_colab.ipynb` 的输出：
1. 合并 HippoDeep 特征表 + OASIS 官方 demographics/CDR 标签表，产出全项目共用的主表。
2. 解析每个受试者的 `aseg.stats`（FreeSurfer 已经算好的海马体积 + 全脑体积/ICV），
   验证 HippoDeep 海马体积是否可信；全脑体积特征直接用 `aseg.stats`（缺失时 fallback 到官方 nWBV）。

全脑体积不再用 SynthSeg 独立计算——SynthSeg 的预训练模型权重需要从 UCL SharePoint 手动下载，
官方链接经常打不开（GitHub issue #116 长期未解决），属于不受控的外部依赖，已放弃。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'openpyxl', 'pandas', 'scipy', 'matplotlib'])

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

DRIVE_ROOT = '/content/drive/MyDrive'
OASIS_DIR  = f'{DRIVE_ROOT}/OASIS_Freesurfer'
XLSX_PATH  = f'{OASIS_DIR}/oasis_cross-sectional-5708aa0a98d82080.xlsx'

OUTPUT_DIR    = f'{DRIVE_ROOT}/alzheimer_anfis_outputs'
FEATURES_CSV  = f'{OUTPUT_DIR}/oasis_features_hippodeep_synthseg.csv'
MASTER_CSV    = f'{OUTPUT_DIR}/oasis_master_features_labels.csv'
RESULTS_DIR   = f'{OUTPUT_DIR}/results'
os.makedirs(RESULTS_DIR, exist_ok=True)

In [ ]:
# ── 读取官方标签表（CDR/MMSE/Age/官方 nWBV），二分类映射 ───────────
# 与 alzheimer_dsnet_oasis.ipynb 的 CDR 处理逻辑一致：
# CDR=NaN（年轻健康对照）→ Nondemented；CDR=0（老年健康对照）→ Nondemented；CDR>0 → Demented

df_labels = pd.read_excel(XLSX_PATH)
df_labels['subject_id'] = df_labels['ID'].astype(str).str.strip()

def cdr_to_label(cdr):
    if pd.isna(cdr) or cdr == 0:
        return 'Nondemented'
    return 'Demented'

df_labels['label'] = df_labels['CDR'].apply(cdr_to_label)
df_labels['cdr_numeric'] = df_labels['CDR'].fillna(0.0)  # ANFIS 回归目标用；NaN(年轻对照)按0处理

label_cols = ['subject_id', 'Age', 'MMSE', 'CDR', 'cdr_numeric', 'label', 'nWBV', 'eTIV']
df_labels = df_labels[label_cols].rename(columns={'Age': 'age', 'MMSE': 'mmse', 'nWBV': 'nwbv_official', 'eTIV': 'etiv_official'})

print(f'标签表共 {len(df_labels)} 行')
print(df_labels['label'].value_counts())

In [ ]:
# ── 合并特征表 + 标签表 ─────────────────────────────────────────
features_df = pd.read_csv(FEATURES_CSV)
master_df = features_df.merge(df_labels, on='subject_id', how='inner')

print(f'特征表 {len(features_df)} 例，标签表 {len(df_labels)} 例，合并后 {len(master_df)} 例')
print(master_df['label'].value_counts())
missing = set(features_df['subject_id']) - set(df_labels['subject_id'])
if missing:
    print(f'\n警告：{len(missing)} 个受试者有特征但标签表里找不到，已在合并时丢弃：', list(missing)[:10])

In [ ]:
# ── 解析 aseg.stats（用 os.walk 而不是 Path.glob，原因同 feature_extraction notebook：
# Colab 挂载的 Google Drive 上 pathlib 递归 glob 经常返回空结果）────────────────────
# Left-Hippocampus / Right-Hippocampus 的 Volume_mm3 在第 4 列；
# 全脑测量在 "# Measure ... BrainSegVol/IntraCranialVol" 注释行里。

def parse_aseg_stats(stats_path: str) -> dict:
    left_hippo = right_hippo = brain_seg_vol = icv = None
    with open(stats_path) as f:
        for line in f:
            if line.startswith('# Measure BrainSeg,'):
                brain_seg_vol = float(line.split(',')[3].strip())
            elif line.startswith('# Measure IntraCranialVol,'):
                icv = float(line.split(',')[3].strip())
            elif not line.startswith('#'):
                parts = line.split()
                if len(parts) < 5:
                    continue
                struct_name = parts[4]
                vol = float(parts[3])
                if struct_name == 'Left-Hippocampus':
                    left_hippo = vol
                elif struct_name == 'Right-Hippocampus':
                    right_hippo = vol
    return {
        'aseg_hippo_left': left_hippo,
        'aseg_hippo_right': right_hippo,
        'aseg_hippo_total': (left_hippo or 0) + (right_hippo or 0) if left_hippo and right_hippo else None,
        'aseg_brain_seg_vol': brain_seg_vol,
        'aseg_icv': icv,
        'aseg_wbv': (brain_seg_vol / icv) if (brain_seg_vol and icv) else None,
    }

print('正在扫描 aseg.stats 文件（受试者数量大时可能需要几分钟）...')
stats_paths = {}
for root, dirs, files in os.walk(OASIS_DIR):
    if 'aseg.stats' in files and os.path.basename(root) == 'stats':
        sid = os.path.basename(os.path.dirname(root))
        stats_paths[sid] = os.path.join(root, 'aseg.stats')
stats_paths = {sid: p for sid, p in stats_paths.items() if sid.endswith('_MR1')}
print(f'找到 {len(stats_paths)} 个受试者的 aseg.stats（这就是我们能做验证、以及全脑体积的主要来源）')

aseg_rows = []
for sid, p in stats_paths.items():
    row = {'subject_id': sid}
    row.update(parse_aseg_stats(p))
    aseg_rows.append(row)
aseg_df = pd.DataFrame(aseg_rows)

master_df = master_df.merge(aseg_df, on='subject_id', how='left')
n_with_gt = master_df['aseg_hippo_total'].notna().sum()
print(f'主表中有 {n_with_gt} / {len(master_df)} 例同时具备 aseg.stats 真值，可用于特征提取质量验证')

In [ ]:
# ── 全脑体积特征 wbv_final：优先用 aseg_wbv（FreeSurfer 直接算的 BrainSegVol/ICV），
# 缺失时 fallback 到官方 nwbv_official（xlsx，全部 416 人都有），保证覆盖率最大化。
master_df['wbv_final'] = master_df['aseg_wbv'].fillna(master_df['nwbv_official'])
n_wbv_from_aseg = master_df['aseg_wbv'].notna().sum()
n_wbv_from_fallback = master_df['wbv_final'].notna().sum() - n_wbv_from_aseg
n_wbv_missing = master_df['wbv_final'].isna().sum()
print(f'wbv_final: {n_wbv_from_aseg} 例来自 aseg_wbv，{n_wbv_from_fallback} 例 fallback 到官方 nWBV，'
      f'{n_wbv_missing} 例两者都缺失')

In [ ]:
# ── 特征提取质量验证：HippoDeep vs aseg.stats 海马体积 ──────────────
val_df = master_df.dropna(subset=['aseg_hippo_total', 'hippo_vol_hippodeep']).copy()
print(f'海马体积验证样本数: {len(val_df)}')

if len(val_df) >= 3:
    r_hippo, p_hippo = stats.pearsonr(val_df['aseg_hippo_total'], val_df['hippo_vol_hippodeep'])
    mae_hippo = (val_df['aseg_hippo_total'] - val_df['hippo_vol_hippodeep']).abs().mean()
    print(f'HippoDeep vs aseg.stats 海马体积: Pearson r={r_hippo:.3f} (p={p_hippo:.2e}), MAE={mae_hippo:.1f} mm^3')
    if r_hippo < 0.8:
        print('  ⚠️ 相关系数低于 0.8 门槛——先检查 HippoDeep 输出单位（mm^3 vs mL）和输入方向/强度归一化是否正确，'
              '不要直接拿这批特征去训练 ANFIS。')
else:
    print('验证样本太少，无法计算相关系数——检查上面 aseg.stats 是否解析成功。')

# Bland-Altman 图
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(val_df['aseg_hippo_total'], val_df['hippo_vol_hippodeep'], alpha=0.6)
lims = [val_df[['aseg_hippo_total', 'hippo_vol_hippodeep']].min().min(),
        val_df[['aseg_hippo_total', 'hippo_vol_hippodeep']].max().max()]
axes[0].plot(lims, lims, 'k--', lw=1)
axes[0].set_xlabel('aseg.stats 海马体积 (mm^3, 真值)')
axes[0].set_ylabel('HippoDeep 海马体积 (mm^3)')
axes[0].set_title('Scatter: HippoDeep vs FreeSurfer')

mean_vol = (val_df['aseg_hippo_total'] + val_df['hippo_vol_hippodeep']) / 2
diff_vol = val_df['hippo_vol_hippodeep'] - val_df['aseg_hippo_total']
axes[1].scatter(mean_vol, diff_vol, alpha=0.6)
axes[1].axhline(diff_vol.mean(), color='r', linestyle='--', label=f'mean diff={diff_vol.mean():.1f}')
axes[1].axhline(diff_vol.mean() + 1.96 * diff_vol.std(), color='gray', linestyle=':')
axes[1].axhline(diff_vol.mean() - 1.96 * diff_vol.std(), color='gray', linestyle=':')
axes[1].set_xlabel('Mean of two methods')
axes[1].set_ylabel('HippoDeep - FreeSurfer')
axes[1].set_title('Bland-Altman: 海马体积')
axes[1].legend()
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/hippo_validation.png', dpi=150)
plt.show()

In [ ]:
# ── 保存最终主表（后续 ANFIS notebook 直接读这一份）────────────────
master_df.to_csv(MASTER_CSV, index=False)
print(f'保存主表: {MASTER_CSV}  ({len(master_df)} 例，其中 {n_with_gt} 例有 aseg.stats 真值，'
      f'{master_df["wbv_final"].notna().sum()} 例有 wbv_final)')
master_df.head()